# Lecture 11 作业：中文文本分类

学号：`20234080203`　姓名：叶平

按 `HW-lecture11.pdf` 要求，使用 Hugging Face Transformers、Datasets 和 PyTorch，完成课件中的中文文本分类实验：数据集 `lansinuote/ChnSentiCorp`，预训练模型 `google-bert/bert-base-chinese`。

## 1. 环境与学号信息

In [2]:
import os
import sys
import hashlib
import torch
import transformers
import datasets
student_id = "20234080203" # 修改为本人学号
student_name = "叶平"	# 修改为本人姓名
conda_env = os.environ.get("CONDA_DEFAULT_ENV", "")
assert conda_env and conda_env != "base", "请先激活独立的 Conda 环境并在该环境启动 Jupyter Kernel"
assert student_id.isdigit() and student_name.strip(), "请正确填写学号和姓名"
# 基于学号生成固定随机种子；后续抽样与划分必须使用该种子
student_seed = int(hashlib.sha256(student_id.encode("utf-8")).hexdigest()[:8], 16)
print("Student ID:", student_id)
print("Student Name:", student_name)
print("Conda Environment:", conda_env)
print("Python Executable:", sys.executable)
print("Python Version:", sys.version.split()[0])
print("PyTorch Version:", torch.__version__)
print("Transformers Version:", transformers.__version__)
print("Datasets Version:", datasets.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("Student Seed:", student_seed)

c:\Users\Administrator\Miniconda3\envs\lecture11\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Student ID: 20234080203
Student Name: 叶平
Conda Environment: lecture11
Python Executable: c:\Users\Administrator\Miniconda3\envs\lecture11\python.exe
Python Version: 3.10.20
PyTorch Version: 2.6.0+cu124
Transformers Version: 5.19.0
Datasets Version: 5.1.0
CUDA Available: True
Device: cuda
Student Seed: 3320517646


## 2. 加载 ChnSentiCorp 数据集

使用 `load_dataset()` 加载课件中的中文情感分类数据，打印 train / validation / test 样本数，并展示一条样本。

In [3]:
from datasets import load_dataset

dataset = load_dataset("lansinuote/ChnSentiCorp")

print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print("Test size:", len(dataset["test"]))
print("Features:", dataset["train"].features)
print("Sample[0]:", dataset["train"][0])

Train size: 9600
Validation size: 1200
Test size: 1200
Features: {'text': Value('string'), 'label': ClassLabel(names=['negative', 'positive'])}
Sample[0]: {'text': '选择珠江花园的原因就是方便，有电动扶梯直接到达海边，周围餐馆、食廊、商场、超市、摊位一应俱全。酒店装修一般，但还算整洁。 泳池在大堂的屋顶，因此很小，不过女儿倒是喜欢。 包的早餐是西式的，还算丰富。 服务吗，一般', 'label': 1}


## 3. Tokenizer 编码示例

使用 `AutoTokenizer` 加载 `google-bert/bert-base-chinese`，展示一条文本的 `input_ids`、`attention_mask` 及解码结果。

In [4]:
from transformers import AutoTokenizer

MODEL_NAME = "google-bert/bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_text = dataset["train"][0]["text"]
encoded = tokenizer(sample_text, truncation=True, max_length=128)
decoded = tokenizer.decode(encoded["input_ids"], skip_special_tokens=False)

print("Original text:", sample_text)
print("input_ids:", encoded["input_ids"])
print("attention_mask:", encoded["attention_mask"])
print("Decoded:", decoded)

Original text: 选择珠江花园的原因就是方便，有电动扶梯直接到达海边，周围餐馆、食廊、商场、超市、摊位一应俱全。酒店装修一般，但还算整洁。 泳池在大堂的屋顶，因此很小，不过女儿倒是喜欢。 包的早餐是西式的，还算丰富。 服务吗，一般
input_ids: [101, 6848, 2885, 4403, 3736, 5709, 1736, 4638, 1333, 1728, 2218, 3221, 3175, 912, 8024, 3300, 4510, 1220, 2820, 3461, 4684, 2970, 1168, 6809, 3862, 6804, 8024, 1453, 1741, 7623, 7667, 510, 7608, 2443, 510, 1555, 1767, 510, 6631, 2356, 510, 3033, 855, 671, 2418, 936, 1059, 511, 6983, 2421, 6163, 934, 671, 5663, 8024, 852, 6820, 5050, 3146, 3815, 511, 3807, 3737, 1762, 1920, 1828, 4638, 2238, 7553, 8024, 1728, 3634, 2523, 2207, 8024, 679, 6814, 1957, 1036, 948, 3221, 1599, 3614, 511, 1259, 4638, 3193, 7623, 3221, 6205, 2466, 4638, 8024, 6820, 5050, 705, 2168, 511, 3302, 1218, 1408, 8024, 671, 5663, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [5]:
import numpy as np

n_sample = 400  # 减少样本数加快运行
n_train = len(dataset["train"])
assert n_sample >= 200, "抽样数量不得少于 200"
assert n_sample <= n_train

rng = np.random.default_rng(student_seed)
train_indices = rng.choice(n_train, size=n_sample, replace=False)
train_subset = dataset["train"].select(train_indices.tolist())

print("Sampled train size:", len(train_subset))
print("First 10 original indices:", train_indices[:10].tolist())
print("Subset sample[0]:", train_subset[0])
label_counts = {
    0: int((np.array(train_subset['label']) == 0).sum()),
    1: int((np.array(train_subset['label']) == 1).sum())
}
print("Label counts:", label_counts)

Sampled train size: 400
First 10 original indices: [2520, 765, 946, 1789, 8394, 7895, 5949, 1763, 3709, 6869]
Subset sample[0]: {'text': '杀毒软件和office2007都是60天试用版 QQ里摄像头使用有点麻烦，要先在设置里更改视频设备为“USB视频设备” 六芯电池突出来有点多 硬盘震动稍稍大了一点 屏幕仰角过大的话容易翻过去', 'label': 0}
Label counts: {0: 201, 1: 199}


## 5. 构建并训练 BERT 二分类模型

参考课件：用 `AutoModel` 抽取 `[CLS]` 特征，接线性层做二分类；预训练参数冻结，只更新分类头。训练仅使用 `train_subset`，测试集不参与训练。

In [6]:
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModel
import time

device = torch.device("cpu")  # 使用CPU避免CUDA问题
MAX_LEN = 128
BATCH_SIZE = 4  # 减小batch size避免显存问题
EPOCHS = 2  # 减少epoch加快运行
LR = 1e-4


def collate_fn(batch):
    texts = [item["text"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
        return_token_type_ids=False,
    )
    encoded["labels"] = labels
    return encoded


class BertClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.pretrained = AutoModel.from_pretrained(MODEL_NAME)
        self.fc = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask, labels=None):
        with torch.no_grad():
            hidden = self.pretrained(
                input_ids=input_ids, attention_mask=attention_mask
            ).last_hidden_state
        logits = self.fc(hidden[:, 0])
        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)
        return loss, logits


train_loader = DataLoader(
    train_subset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
model = BertClassifier().to(device)
optimizer = torch.optim.Adam(model.fc.parameters(), lr=LR)

print("Device:", device)
print("Train batches:", len(train_loader))
print("Adjustments: n_sample=400, max_length=128, batch_size=4, epochs=2, freeze BERT")

model.train()
for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    epoch_start = time.time()
    for batch_idx, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        loss, logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch["labels"].size(0)
        total_correct += (logits.argmax(dim=1) == batch["labels"]).sum().item()
        total_count += batch["labels"].size(0)
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(train_loader)}, loss={loss.item():.4f}")
    epoch_time = time.time() - epoch_start
    print(
        f"Epoch {epoch}/{EPOCHS}  loss={total_loss / total_count:.4f}  "
        f"train_acc={total_correct / total_count:.4f}  time={epoch_time:.1f}s"
    )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 800.65it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cpu
Train batches: 100
Adjustments: n_sample=400, max_length=128, batch_size=4, epochs=2, freeze BERT
  Batch 10/100, loss=0.6783
  Batch 20/100, loss=0.7631
  Batch 30/100, loss=0.6652
  Batch 40/100, loss=0.5534
  Batch 50/100, loss=0.5248
  Batch 60/100, loss=0.5808
  Batch 70/100, loss=0.7606
  Batch 80/100, loss=0.6593
  Batch 90/100, loss=0.5153
  Batch 100/100, loss=0.6541
Epoch 1/2  loss=0.6289  train_acc=0.6600  time=27.1s
  Batch 10/100, loss=0.5289
  Batch 20/100, loss=0.3897
  Batch 30/100, loss=0.6479
  Batch 40/100, loss=0.5201
  Batch 50/100, loss=0.4237
  Batch 60/100, loss=0.4879
  Batch 70/100, loss=0.4377
  Batch 80/100, loss=0.5254
  Batch 90/100, loss=0.5760
  Batch 100/100, loss=0.4579
Epoch 2/2  loss=0.5329  train_acc=0.8200  time=28.9s


## 6. 测试集准确率

关闭梯度计算，在**完整测试集**上评估。测试集未用于训练。

In [ ]:
test_loader = DataLoader(
    dataset["test"], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        _, logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        pred = logits.argmax(dim=1)
        correct += (pred == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

accuracy = correct / total
print("Test size:", total)
print("Test accuracy:", accuracy)
print(f"Accuracy: {accuracy:.4f}")

Test size: 1200
Test accuracy: 0.8416666666666667
Accuracy: 0.8417


: 